In [67]:
import pandas as pd

# Mock source documents
documents = [
    "The city council approved a plan to add 20 electric buses and reduce fares for students.",
    "A new study shows that regular exercise reduces the risk of heart disease by 30%.",
    "The company announced a new AI-powered product aimed at improving customer support.",
    "Heavy rainfall caused flooding in several مناطق, displacing hundreds of residents.",
    "Scientists discovered a new species of marine الحياة in the Pacific Ocean."
]

# Mock reference summaries
reference_summaries = [
    "The council approved electric buses and student fare reductions.",
    "Exercise significantly lowers heart disease risk.",
    "A company launched an AI tool for customer support.",
    "Flooding displaced residents after heavy rainfall.",
    "A new marine species was discovered in the Pacific."
]

# Mock fine-tuned model summaries (5 models)
models = ["ft_model_A", "ft_model_B", "ft_model_C", "ft_model_D", "ft_model_E"]

# Generate mock summaries (slightly varied quality)
def generate_mock_summary(doc, model_id):
    return f"{model_id}: {doc[:80]}..."

# Build dataset
rows = []

for doc_id, (doc, ref) in enumerate(zip(documents, reference_summaries), start=1):
    for model in models:
        rows.append({
            "doc_id": f"doc_{doc_id}",
            "source_text": doc,
            "model_id": model,
            "summary_text": generate_mock_summary(doc, model),
            "reference_summary": ref
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Show sample

# Save to CSV (optional)
df.to_csv("mock_summarization_dataset.csv", index=False)
df.head(10)


,doc_id,source_text,model_id,summary_text,reference_summary
0,doc_1,The city council approved a plan to add 20 ele...,ft_model_A,ft_model_A: The city council approved a plan t...,The council approved electric buses and studen...
1,doc_1,The city council approved a plan to add 20 ele...,ft_model_B,ft_model_B: The city council approved a plan t...,The council approved electric buses and studen...
2,doc_1,The city council approved a plan to add 20 ele...,ft_model_C,ft_model_C: The city council approved a plan t...,The council approved electric buses and studen...
3,doc_1,The city council approved a plan to add 20 ele...,ft_model_D,ft_model_D: The city council approved a plan t...,The council approved electric buses and studen...
4,doc_1,The city council approved a plan to add 20 ele...,ft_model_E,ft_model_E: The city council approved a plan t...,The council approved electric buses and studen...
5,doc_2,A new study shows that regular exercise reduce...,ft_model_A,ft_model_A: A new study shows that regular exe...,Exercise significantly lowers heart disease risk.
6,doc_2,A new study shows that regular exercise reduce...,ft_model_B,ft_model_B: A new study shows that regular exe...,Exercise significantly lowers heart disease risk.
7,doc_2,A new study shows that regular exercise reduce...,ft_model_C,ft_model_C: A new study shows that regular exe...,Exercise significantly lowers heart disease risk.
8,doc_2,A new study shows that regular exercise reduce...,ft_model_D,ft_model_D: A new study shows that regular exe...,Exercise significantly lowers heart disease risk.
9,doc_2,A new study shows that regular exercise reduce...,ft_model_E,ft_model_E: A new study shows that regular exe...,Exercise significantly lowers heart disease risk.


In [1]:
import os
from pathlib import Path

import pandas as pd

# Data moved out of the repo (2026-06) into the shared OneDrive folder. Override with
# CHECKPOINT_SELECTION_DATA_DIR if your OneDrive root or the dataset snapshot name differs.
DATA_ROOT = Path(
    os.environ.get("CHECKPOINT_SELECTION_DATA_DIR")
    or (
        Path(os.environ.get("ONEDRIVE", str(Path.home() / "OneDrive")))
        / "Shared"
        / "Demokratibasen-UiB-Ide"
        / "EvaluationDatasets"
        / "CheckpointSelection"
        / "Data_202606"
    )
)


def resolve_eval_data_dir() -> Path:
    data_root_eval = DATA_ROOT / "eval"
    if data_root_eval.is_dir():
        return data_root_eval
    cwd = Path.cwd()
    for candidate in (cwd / "Data" / "eval", cwd.parent / "Data" / "eval"):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find eval data. Expected at {data_root_eval} "
        "(set ONEDRIVE or CHECKPOINT_SELECTION_DATA_DIR if that's not your OneDrive layout), "
        "or a legacy Data/eval under cwd."
    )


def jsonl_to_long_df(eval_dir: Path) -> pd.DataFrame:
    """One row per (document, model): same columns as the toy `df` in the previous cell."""
    files = sorted(eval_dir.glob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No .jsonl files under {eval_dir}")

    rows: list[dict] = []
    baseline_input: pd.Series | None = None

    for path in files:
        model_id = path.stem
        part = pd.read_json(path, lines=True)
        required = {"input_text", "prompt", "reference", "prediction"}
        missing = required - set(part.columns)
        if missing:
            raise ValueError(f"{path.name}: missing columns {missing}")

        part = part.copy()
        part["prediction"] = part["prediction"].fillna("").astype(str)

        if baseline_input is None:
            baseline_input = part["input_text"].astype(str).reset_index(drop=True)
        else:
            if not part["input_text"].astype(str).reset_index(drop=True).equals(baseline_input):
                raise ValueError(
                    f"{path.name}: `input_text` rows do not match the first file (document order)."
                )

        for i, r in part.iterrows():
            rows.append(
                {
                    "doc_id": f"doc_{i + 1}",
                    "source_text": r["input_text"],
                    "model_id": model_id,
                    "summary_text": r["prediction"],
                    "reference_summary": r["reference"],
                }
            )

    return pd.DataFrame(rows)


EVAL_DATA_DIR = resolve_eval_data_dir()
df_eval = jsonl_to_long_df(EVAL_DATA_DIR)

models_eval = sorted(df_eval["model_id"].unique())
n_docs = df_eval["doc_id"].nunique()

print(EVAL_DATA_DIR.resolve())
print(f"rows={len(df_eval)}  documents={n_docs}  models={len(models_eval)}")
print(df_eval.head(3))

# Downstream cells expect `df` and `models`. Uncomment to analyze real eval runs instead of the toy example:
df, models = df_eval, models_eval

/Users/baharehfatemi/Desktop/demokratibasen/Data/eval
rows=11000  documents=1000  models=11
  doc_id                                        source_text  \
0  doc_1  Dokument: Vedtak - Gbnr 26/142 - Botnaneset 27...   
1  doc_2  Dokument: Saksprotokoll BR1, 22052025, Sak 129...   
2  doc_3  Dokument: Kvalitetsplan for oppvekst og kultur...   

                                            model_id  \
0  eurollm-9B-Instruct-checkpoint-5000-inputs-ref...   
1  eurollm-9B-Instruct-checkpoint-5000-inputs-ref...   
2  eurollm-9B-Instruct-checkpoint-5000-inputs-ref...   

                                        summary_text  \
0  *Note: The provided document is a Norwegian mu...   
1                                                      
2                                                      

                                   reference_summary  
0  Kinn kommune har godkjent søknaden fra Fjord B...  
1  Byrådet har behandlet statusmeldingen for sosi...  
2  Etne kommune har utarbeidet en ny kva

In [2]:
import itertools

import numpy as np

RNG = np.random.default_rng(42)  # change seed for a different split


def build_pairs_table(long_df: pd.DataFrame, n_pairs: int = 4, rng: np.random.Generator = RNG) -> pd.DataFrame:
    """Sample `n_pairs` distinct model pairs per doc; left/right order randomized per pair."""
    rows = []
    for doc_id, g in long_df.groupby("doc_id", sort=False):
        by_model = g.set_index("model_id")["summary_text"].to_dict()
        models_here = list(by_model.keys())
        if len(models_here) < 2:
            continue
        all_pairs = list(itertools.combinations(models_here, 2))
        k = min(n_pairs, len(all_pairs))
        idx = rng.choice(len(all_pairs), size=k, replace=False)
        chosen_pairs = [all_pairs[i] for i in idx]

        for a, b in chosen_pairs:
            if rng.random() < 0.5:
                left, right = a, b
            else:
                left, right = b, a
            rows.append(
                {
                    "doc_id": doc_id,
                    "left": left,
                    "right": right,
                    "sumleft": by_model[left],
                    "sumright": by_model[right],
                }
            )
    return pd.DataFrame(rows)


pairs_table = build_pairs_table(df, n_pairs=4)
pairs_table.head(8)

,doc_id,left,right,sumleft,sumright
0,doc_1,gemma-2b-apptainer-checkpoint-5000-inputs-refs...,normistral-7b-apptainer-checkpoint-5000-inputs...,Vedtak om tilbygg til et eksisterende næringsb...,I et brev fra arealforvaltningsenheten informe...
1,doc_1,llama-3.1-8b-instruct-apptainer-checkpoint-500...,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,Kinn kommune har gitt tillatelse til Fjord Bas...,*Note: The provided document is a Norwegian mu...
2,doc_1,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,Tilbygget vil inneholde kontrollrom og arbeids...,Dokumentet beskriver vedtaket fra Kinn kommune...
3,doc_1,normistral-11b-apptainer-checkpoint-5000-input...,llama-3.1-8b-instruct-apptainer-checkpoint-500...,Kinn kommune har innvilget tillatelse til et n...,Kinn kommune har gitt tillatelse til Fjord Bas...
4,doc_2,normistral-11b-apptainer-checkpoint-5000-input...,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,Byrådet har presentert og innstilt til bystyre...,
5,doc_2,normistral-11b-apptainer-checkpoint-5000-input...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,Byrådet har presentert og innstilt til bystyre...,Dokumentet inneholder en statusmelding for sos...
6,doc_2,normistral-7b-apptainer-checkpoint-5000-inputs...,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,I byrådsmøtet den 36 ble det besluttet at byst...,Hvilke strategier bruker du for å holde deg or...
7,doc_2,gemma-7b-apptainer-checkpoint-5000-inputs-refs...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,Byrådets behandling av den aktuelle saksordnin...,Dokumentet inneholder en statusmelding for sos...


In [3]:
pairs_table.shape

(4000, 5)

In [4]:
import hashlib
from typing import Callable, Dict, Mapping, Tuple

import numpy as np
import pandas as pd

EVAL_DIMENSIONS = ("faithfulness", "correctness", "completeness")
JUDGES = ("human", "llm_1", "llm_2", "llm_3")

# Stub judge only: probability of a tie label. Set to 0.0 for no ties in mock runs.
MOCK_TIE_PROB = 0.1


def _is_tie_row(row: Mapping) -> bool:
    """True if the judge did not pick left or right (tie)."""
    if row.get("choice_side") == "tie":
        return True
    return pd.isna(row.get("chosen"))


def _rng_for_judge_dimension(judge_id: str, dimension: str, base_seed: int = 42) -> np.random.Generator:
    digest = hashlib.sha256(f"{judge_id}:{dimension}:{base_seed}".encode()).digest()
    seed = int.from_bytes(digest[:8], "big") % (2**63 - 1)
    return np.random.default_rng(seed)


def mock_evaluate_pair(
    row: Mapping,
    dimension: str,
    judge_id: str,
    rng: np.random.Generator,
) -> Dict[str, object]:
    """Stub judge: random A/B or tie for this dimension. Replace with human or LLM call."""
    _ = (dimension, judge_id)
    if rng.random() < MOCK_TIE_PROB:
        return {"choice_side": "tie", "chosen": pd.NA, "rationale": ""}
    pick_left = rng.random() < 0.5
    return {
        "choice_side": "left" if pick_left else "right",
        "chosen": row["left"] if pick_left else row["right"],
        "rationale": "",
    }


EvaluateFn = Callable[[Mapping, str, str, np.random.Generator], Dict[str, object]]


def attach_doc_context(pairs_df: pd.DataFrame, long_df: pd.DataFrame) -> pd.DataFrame:
    meta = long_df.groupby("doc_id", sort=False).first()[["source_text", "reference_summary"]]
    return pairs_df.merge(meta, on="doc_id", how="left")


def build_geval_tables(
    pairs_df: pd.DataFrame,
    long_df: pd.DataFrame,
    dimensions: Tuple[str, ...] = EVAL_DIMENSIONS,
    judges: Tuple[str, ...] = JUDGES,
    evaluate_fn: EvaluateFn = mock_evaluate_pair,
    base_seed: int = 42,
) -> Dict[Tuple[str, str], pd.DataFrame]:
    """One DataFrame per (judge_id, dimension): pair rows plus chosen, choice_side (left/right/tie), rationale."""
    ctx = attach_doc_context(pairs_df, long_df)
    out: Dict[Tuple[str, str], pd.DataFrame] = {}
    for judge_id in judges:
        for dimension in dimensions:
            rng = _rng_for_judge_dimension(judge_id, dimension, base_seed=base_seed)
            rows = []
            for _, row in ctx.iterrows():
                judgment = evaluate_fn(row, dimension, judge_id, rng)
                rows.append({**row.to_dict(), **judgment})
            out[(judge_id, dimension)] = pd.DataFrame(rows)
    return out


geval_tables = build_geval_tables(pairs_table, df)
geval_by_judge = {j: {d: geval_tables[j, d] for d in EVAL_DIMENSIONS} for j in JUDGES}

list(geval_tables.keys())[:4], geval_tables[("human", "faithfulness")].head(2)

#print the first 10 rows of the geval_tables
geval_tables[("human", "faithfulness")][["doc_id", "left", "right", "choice_side", "chosen", "rationale"]].head(30)


,doc_id,left,right,choice_side,chosen,rationale
0,doc_1,gemma-2b-apptainer-checkpoint-5000-inputs-refs...,normistral-7b-apptainer-checkpoint-5000-inputs...,left,gemma-2b-apptainer-checkpoint-5000-inputs-refs...,
1,doc_1,llama-3.1-8b-instruct-apptainer-checkpoint-500...,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,tie,<NA>,
2,doc_1,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,right,llama-2-13b-chat-norwegian-apptainer-checkpoin...,
3,doc_1,normistral-11b-apptainer-checkpoint-5000-input...,llama-3.1-8b-instruct-apptainer-checkpoint-500...,right,llama-3.1-8b-instruct-apptainer-checkpoint-500...,
4,doc_2,normistral-11b-apptainer-checkpoint-5000-input...,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,right,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,
5,doc_2,normistral-11b-apptainer-checkpoint-5000-input...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,right,llama-2-13b-chat-norwegian-apptainer-checkpoin...,
6,doc_2,normistral-7b-apptainer-checkpoint-5000-inputs...,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,right,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,
7,doc_2,gemma-7b-apptainer-checkpoint-5000-inputs-refs...,llama-2-13b-chat-norwegian-apptainer-checkpoin...,right,llama-2-13b-chat-norwegian-apptainer-checkpoin...,
8,doc_3,llama-3.1-8b-instruct-apptainer-checkpoint-500...,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,left,llama-3.1-8b-instruct-apptainer-checkpoint-500...,
9,doc_3,llama-2-13b-chat-norwegian-apptainer-checkpoin...,gemma-2-9b-checkpoint-5000-inputs-refs-preds-e...,right,gemma-2-9b-checkpoint-5000-inputs-refs-preds-e...,


In [5]:
import json
from pathlib import Path
from typing import Dict, Tuple


def resolve_geval_export_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == ".deepeval":
        return cwd / "geval_exports"
    return cwd / ".deepeval" / "geval_exports"


def save_geval_json(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    pairs_df: pd.DataFrame,
    long_df: pd.DataFrame,
    export_dir: Path | None = None,
) -> Path:
    """Write pairs, long-form data, and each judge×dimension table as JSON (records, indented)."""
    out = export_dir or resolve_geval_export_dir()
    out.mkdir(parents=True, exist_ok=True)
    manifest: dict = {"export_dir": str(out.resolve()), "files": []}

    def write_df(stem: str, frame: pd.DataFrame) -> None:
        path = out / f"{stem}.json"
        frame.to_json(path, orient="records", indent=2, force_ascii=False)
        manifest["files"].append({"file": path.name, "rows": len(frame)})

    write_df("pairs_table", pairs_df)
    write_df("summarization_long", long_df)
    for (judge_id, dimension), tbl in geval_tables.items():
        stem = f"geval__{judge_id}__{dimension}"
        write_df(stem, tbl)

    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return out


export_path = save_geval_json(geval_tables, pairs_table, df)
export_path, (export_path / "manifest.json").read_text(encoding="utf-8")[:400]

(PosixPath('/Users/baharehfatemi/Desktop/demokratibasen/.deepeval/geval_exports'),
 '{\n  "export_dir": "/Users/baharehfatemi/Desktop/demokratibasen/.deepeval/geval_exports",\n  "files": [\n    {\n      "file": "pairs_table.json",\n      "rows": 4000\n    },\n    {\n      "file": "summarization_long.json",\n      "rows": 11000\n    },\n    {\n      "file": "geval__human__faithfulness.json",\n      "rows": 4000\n    },\n    {\n      "file": "geval__human__correctness.json",\n      "rows": 4000\n    ')

In [6]:
from collections import defaultdict
from typing import Dict, Iterable, Tuple

# Human vs LLM judge groups (edit if your ids differ)
HUMAN_JUDGES: Tuple[str, ...] = ("human",)
LLM_JUDGES: Tuple[str, ...] = tuple(j for j in JUDGES if j not in HUMAN_JUDGES)


def wins_and_opportunities_for_group(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    judge_ids: Iterable[str],
    dimensions: Tuple[str, ...],
) -> Tuple[Dict[str, int], Dict[str, int]]:
    """Per model: fractional wins (ties count as 0.5 each) vs pair appearances."""
    wins: dict[str, float] = defaultdict(float)
    opps: dict[str, int] = defaultdict(int)
    for j in judge_ids:
        for d in dimensions:
            tbl = geval_tables[(j, d)]
            for _, row in tbl.iterrows():
                opps[row["left"]] += 1
                opps[row["right"]] += 1
                if _is_tie_row(row):
                    wins[row["left"]] += 0.5
                    wins[row["right"]] += 0.5
                else:
                    wins[row["chosen"]] += 1.0
    return wins, opps


def win_rate_table_paper(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    model_order: Iterable[str] | None = None,
    dimensions: Tuple[str, ...] = EVAL_DIMENSIONS,
) -> pd.DataFrame:
    w_h, o_h = wins_and_opportunities_for_group(geval_tables, HUMAN_JUDGES, dimensions)
    w_l, o_l = wins_and_opportunities_for_group(geval_tables, LLM_JUDGES, dimensions)
    models = list(model_order) if model_order is not None else sorted(set(o_h) | set(o_l))
    rows = []
    for m in models:
        oh, ol = o_h.get(m, 0), o_l.get(m, 0)
        rows.append(
            {
                "model": m,
                "n_pairwise_human": oh,
                "win_rate_human": (w_h[m] / oh) if oh else float("nan"),
                "n_pairwise_llm": ol,
                "win_rate_llm_pooled": (w_l[m] / ol) if ol else float("nan"),
            }
        )
    return pd.DataFrame(rows)


win_rates_paper = win_rate_table_paper(geval_tables, model_order=models)
win_rates_paper

_export = resolve_geval_export_dir()
win_rates_paper.to_csv(_export / "win_rates_by_model.csv", index=False)
win_rates_paper.to_latex(
    buf=_export / "win_rates_by_model.tex",
    index=False,
    float_format="%.3f".format,
    caption="Pairwise win rates by judge type (human vs pooled LLM judges).",
    label="tab:winrates",
)

# Markdown (GitHub-style pipe table; no extra packages)
_wr = win_rates_paper.copy()
for _c in ("win_rate_human", "win_rate_llm_pooled"):
    _wr[_c] = _wr[_c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "—")
_cols = list(_wr.columns)
_md_lines = [
    "# Pairwise win rates by model",
    "",
    "Human judges: all dimensions pooled. LLM: pooled over `llm_1`, `llm_2`, `llm_3`.",
    "",
    "| " + " | ".join(_cols) + " |",
    "| " + " | ".join("---" for _ in _cols) + " |",
]
for _, _r in _wr.iterrows():
    _md_lines.append("| " + " | ".join(str(_r[c]) for c in _cols) + " |")
_md_lines.append("")
(_export / "win_rates_by_model.md").write_text("\n".join(_md_lines), encoding="utf-8")

_export / "win_rates_by_model.md"

PosixPath('/Users/baharehfatemi/Desktop/demokratibasen/.deepeval/geval_exports/win_rates_by_model.md')

In [7]:
from typing import Dict, Iterable, Tuple


def _models_in_dimension(geval_tables: Dict[Tuple[str, str], pd.DataFrame], dimension: str) -> list[str]:
    tbl = geval_tables[("human", dimension)]
    return sorted(set(tbl["left"]) | set(tbl["right"]))


def win_rate_matrix_by_dimension(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    dimension: str,
    judges: Tuple[str, ...] = JUDGES,
    model_order: Iterable[str] | None = None,
) -> pd.DataFrame:
    """Rows = models; columns = {judge}_win_rate for one evaluation dimension (ties = 0.5 win each)."""
    mo = list(model_order) if model_order is not None else _models_in_dimension(geval_tables, dimension)
    col_names = [f"{j}_win_rate" for j in judges]
    rows = []
    for m in mo:
        rec = {"model": m}
        for j, col in zip(judges, col_names):
            tbl = geval_tables[(j, dimension)]
            opps = ((tbl["left"] == m) | (tbl["right"] == m)).sum()
            w = 0.0
            for _, row in tbl.iterrows():
                if _is_tie_row(row):
                    if row["left"] == m or row["right"] == m:
                        w += 0.5
                elif row["chosen"] == m:
                    w += 1.0
            rec[col] = (w / opps) if opps else float("nan")
        rows.append(rec)
    return pd.DataFrame(rows)


def markdown_win_rate_tables_by_dimension(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    dimensions: Tuple[str, ...] = EVAL_DIMENSIONS,
    judges: Tuple[str, ...] = JUDGES,
    model_order: Iterable[str] | None = None,
) -> str:
    mo = list(model_order) if model_order is not None else _models_in_dimension(geval_tables, dimensions[0])
    parts = []
    for dim in dimensions:
        mat = win_rate_matrix_by_dimension(geval_tables, dim, judges, mo)
        parts.append(f"### {dim.capitalize()}\n")
        cols = ["model"] + [f"{j}_win_rate" for j in judges]
        header = "| " + " | ".join(cols) + " |"
        sep = "| " + " | ".join("---" for _ in cols) + " |"
        parts.extend([header, sep])
        for _, r in mat.iterrows():
            cells = [r["model"]] + [f"{r[c]:.3f}" if pd.notna(r[c]) else "—" for c in cols[1:]]
            parts.append("| " + " | ".join(cells) + " |")
        parts.append("")
    return "\n".join(parts)


_wr_dim_md = markdown_win_rate_tables_by_dimension(geval_tables, model_order=models)
print(_wr_dim_md)

_export_dim = resolve_geval_export_dir()
(_export_dim / "win_rates_by_dimension.md").write_text(_wr_dim_md, encoding="utf-8")
_export_dim / "win_rates_by_dimension.md"

### Faithfulness

| model | human_win_rate | llm_1_win_rate | llm_2_win_rate | llm_3_win_rate |
| --- | --- | --- | --- | --- |
| eurollm-9B-Instruct-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.493 | 0.503 | 0.497 | 0.495 |
| gemma-2-9b-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.497 | 0.524 | 0.474 | 0.483 |
| gemma-2b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.504 | 0.519 | 0.487 | 0.534 |
| gemma-7b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.499 | 0.481 | 0.497 | 0.488 |
| llama-2-13b-chat-norwegian-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.498 | 0.496 | 0.489 | 0.490 |
| llama-3.1-8b-instruct-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.505 | 0.472 | 0.492 | 0.488 |
| nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.478 | 0.493 | 0.483 | 0.514 |
| normistral-11b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000 | 0.511 | 0.494 | 0.538 | 0.504 |
| normistral-7b

PosixPath('/Users/baharehfatemi/Desktop/demokratibasen/.deepeval/geval_exports/win_rates_by_dimension.md')

In [8]:
from typing import Dict, Tuple

import numpy as np

try:
    from scipy.optimize import minimize
except ImportError as e:
    raise ImportError("Bradley–Terry MLE needs scipy (`pip install scipy`).") from e


def win_matrix_from_geval(tbl: pd.DataFrame, model_order: list[str]) -> np.ndarray:
    """Directed counts W[i,j] = strength of i over j; decisive +1; tie splits 0.5 each way (BT approximation)."""
    idx = {m: i for i, m in enumerate(model_order)}
    k = len(model_order)
    w = np.zeros((k, k), dtype=float)
    for _, row in tbl.iterrows():
        left, right = row["left"], row["right"]
        i, j = idx[left], idx[right]
        if _is_tie_row(row):
            w[i, j] += 0.5
            w[j, i] += 0.5
        else:
            win = row["chosen"]
            lose = right if win == left else left
            w[idx[win], idx[lose]] += 1.0
    return w


def neg_log_lik_bradley_terry(beta_free: np.ndarray, w: np.ndarray, ref_idx: int = 0) -> float:
    beta = np.insert(beta_free, ref_idx, 0.0)
    k = w.shape[0]
    ll = 0.0
    for i in range(k):
        for j in range(k):
            n = w[i, j]
            if n <= 0:
                continue
            ll += n * (beta[i] - np.log(np.exp(beta[i]) + np.exp(beta[j])))
    return -float(ll)


def fit_bradley_terry(
    w: np.ndarray,
    *,
    ref_idx: int = 0,
) -> tuple[np.ndarray, object]:
    """MLE with sum-to-zero on beta via fixing beta[ref_idx] = 0."""
    k = w.shape[0]
    x0 = np.zeros(k - 1)
    res = minimize(
        neg_log_lik_bradley_terry,
        x0,
        args=(w, ref_idx),
        method="L-BFGS-B",
    )
    beta = np.insert(np.asarray(res.x, dtype=float), ref_idx, 0.0)
    return beta, res


def bradley_terry_long_table(
    geval_tables: Dict[Tuple[str, str], pd.DataFrame],
    dimensions: Tuple[str, ...] = EVAL_DIMENSIONS,
    judges: Tuple[str, ...] = JUDGES,
    model_order: list[str] | None = None,
    ref_model: str | None = None,
) -> pd.DataFrame:
    mo = list(model_order) if model_order is not None else _models_in_dimension(geval_tables, dimensions[0])
    ref_idx = mo.index(ref_model) if ref_model is not None else 0
    rows = []
    for dim in dimensions:
        for judge in judges:
            tbl = geval_tables[(judge, dim)]
            w = win_matrix_from_geval(tbl, mo)
            if (w + w.T).sum() == 0:
                continue
            beta, res = fit_bradley_terry(w, ref_idx=ref_idx)
            theta = np.exp(beta)
            for m, b, t in zip(mo, beta, theta):
                rows.append(
                    {
                        "dimension": dim,
                        "judge": judge,
                        "model": m,
                        "beta": b,
                        "theta": t,
                        "optimizer_success": res.success,
                        "n_comparisons": int((w + w.T).sum() / 2),
                    }
                )
    return pd.DataFrame(rows)


_bt_long = bradley_terry_long_table(geval_tables, model_order=models, ref_model=models[0])
_bt_export = resolve_geval_export_dir()
_bt_long.to_csv(_bt_export / "bradley_terry_long.csv", index=False)

# Wide view: one column per judge for a single dimension (example: faithfulness, theta scale)
_pivot_f = _bt_long[_bt_long["dimension"] == "faithfulness"].pivot(
    index="model", columns="judge", values="theta"
)
_pivot_f.columns = [f"{c}_theta" for c in _pivot_f.columns]
print("Faithfulness — theta (relative scale, ref = first model beta=0):\n", _pivot_f.round(4), sep="")

_bt_long.head(12)

Faithfulness — theta (relative scale, ref = first model beta=0):
                                                    human_theta  llm_1_theta  \
model                                                                          
eurollm-9B-Instruct-checkpoint-5000-inputs-refs...       1.0000       1.0000   
gemma-2-9b-checkpoint-5000-inputs-refs-preds-ex...       1.0168       1.0791   
gemma-2b-apptainer-checkpoint-5000-inputs-refs-...       1.0416       1.0633   
gemma-7b-apptainer-checkpoint-5000-inputs-refs-...       1.0227       0.9197   
llama-2-13b-chat-norwegian-apptainer-checkpoint...       1.0165       0.9746   
llama-3.1-8b-instruct-apptainer-checkpoint-5000...       1.0433       0.8910   
nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-re...       0.9486       0.9602   
normistral-11b-apptainer-checkpoint-5000-inputs...       1.0648       0.9650   
normistral-7b-apptainer-checkpoint-5000-inputs-...       1.0946       1.0416   
normistral-7b-instruct-apptainer-checkpoint-500...     

,dimension,judge,model,beta,theta,optimizer_success,n_comparisons
0,faithfulness,human,eurollm-9B-Instruct-checkpoint-5000-inputs-ref...,0.000000,1.000000,True,4000
1,faithfulness,human,gemma-2-9b-checkpoint-5000-inputs-refs-preds-e...,0.016615,1.016754,True,4000
2,faithfulness,human,gemma-2b-apptainer-checkpoint-5000-inputs-refs...,0.040804,1.041648,True,4000
3,faithfulness,human,gemma-7b-apptainer-checkpoint-5000-inputs-refs...,0.022487,1.022742,True,4000
4,faithfulness,human,llama-2-13b-chat-norwegian-apptainer-checkpoin...,0.016363,1.016498,True,4000
5,faithfulness,human,llama-3.1-8b-instruct-apptainer-checkpoint-500...,0.042395,1.043306,True,4000
6,faithfulness,human,nb-gpt-j-6b-apptainer-checkpoint-5000-inputs-r...,-0.052746,0.948621,True,4000
7,faithfulness,human,normistral-11b-apptainer-checkpoint-5000-input...,0.062819,1.064834,True,4000
8,faithfulness,human,normistral-7b-apptainer-checkpoint-5000-inputs...,0.090418,1.094631,True,4000
9,faithfulness,human,normistral-7b-instruct-apptainer-checkpoint-50...,0.015334,1.015452,True,4000
